# Phase 2 models: do charts and pageviews predict ticket demand?

This notebook walks through the Phase 2 results. Every number is read from the tables the model stage saves in `reports/tables/`, so the notebook stays quick and matches the code exactly. To rebuild everything from the database:

```
python -m concert_demand.ingest.cpi
python -m concert_demand.models
```

**Design in one paragraph.** Shows on one tour are near-copies, so whole tours go to one side of a time split: train on tours starting before 2019, test on tours starting in 2019, and use tours starting after the COVID shutdown (2022-25) as a second holdout. Signals are measured six months before each show. Predictors enter in blocks (controls, then charts, pageviews, career chart history), and each block is judged by how much it improves AUC on held-out tours. Intervals come from resampling whole tours. Coefficients come from a separate model fit on all years with year dummies and standard errors clustered by artist.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from concert_demand.models.sellout import TABLES_DIR

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

# Reference palette: two categorical slots for the two test sets, plus text and grid inks.
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SPLIT_COLOR = {"test_2019": BLUE, "holdout_post": ORANGE}
SPLIT_LABEL = {"test_2019": "2019 test (36 tours)", "holdout_post": "2022-25 holdout (93 tours)"}

plt.rcParams.update({"axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(name):
    return pd.read_csv(TABLES_DIR / f"{name}.csv")

## 1. Part 1: does each block help predict sellouts?

AUC is the chance a randomly picked sold-out show is ranked above a show that didn't sell out (0.5 is a coin flip). The bars are 95% intervals from resampling tours.

In [ ]:
ladders = table("sellout_ladders")
main = ladders[ladders["ladder"] == "charts_first"]

fig, ax = plt.subplots(figsize=(7, 3.6))
steps = list(dict.fromkeys(main["added_block"]))
for offset, (split, rows) in zip([-0.1, 0.1], main.groupby("test_split", sort=False)):
    x = np.arange(len(rows)) + offset
    ax.errorbar(x, rows["auc"], yerr=[rows["auc"] - rows["auc_low"], rows["auc_high"] - rows["auc"]],
                fmt="o", ms=8, lw=2, capsize=0, color=SPLIT_COLOR[split], label=SPLIT_LABEL[split])
    # Direct label in the open space right of the last step, clear of the intervals.
    ax.text(len(rows) - 1 + 0.35, rows["auc"].iloc[-1], SPLIT_LABEL[split],
            va="center", color=INK, fontsize=9)
ax.axhline(0.5, color=MUTED, lw=1, ls="--")
ax.set_xticks(range(len(steps)), ["controls", "+ charts", "+ pageviews", "+ career"])
ax.set_ylabel("Test AUC")
ax.set_title("Signals add little on top of the controls", loc="left", color=INK)
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.set_xlim(-0.4, len(steps) + 0.6)
plt.tight_layout()

main[["test_split", "added_block", "test_tours", "auc", "auc_gain", "gain_low", "gain_high",
      "mean_predicted", "actual_rate"]].round(3)

**Reading it:** AUC stays between 0.52 and 0.62 at every step. Only career chart history on the 2022-25 holdout adds a gain whose interval clears zero (+0.037). The model predicts about 61-65% sold out against 82% actual, because the sellout rate rises every year and the predictive model has no year term (Matt's choice, so that the test can't lean on a year it never saw).

For scale: scoring each show by its own tour's sellout rate, which cheats on purpose, reaches an AUC of about 0.91. Differences between tours explain most of who sells out; the signals just don't capture them.

## 2. Chart signal vs. pageview signal

"Alone" is a block's gain right after the controls; "unique" is its gain after the other signal is already in, so it's what that signal adds that the other doesn't. The Wald p-value tests the whole block at once in the explanatory model.

In [ ]:
table("signal_comparison").round(3)

**Reading it:** neither signal adds anything the other doesn't, and neither block is significant in the explanatory model (charts p = 0.27, pageviews p = 0.37). Pageviews alone help on 2019 only. Spotify streaming (2017-21 shows) adds nothing on top of charts and pageviews.

## 3. Leakage checks: same-show price and number of nights

Both are known only after a show starts selling: price moves with demand (dynamic pricing), and buyers add nights when early ones sell out. Each is added on top of the model to see how much it inflates accuracy.

In [ ]:
leaks = ladders[ladders["added_block"].isin(["show_price", "nights"])]
leaks[["ladder", "test_split", "added_block", "test_tours", "auc_gain", "gain_low", "gain_high"]].round(3)

**Reading it:** on 2022-25 each adds about +0.035 AUC with intervals above zero, as large as any real signal. That is why both stay out of the main model, and why the main model uses the previous tour's price instead.

## 4. Room size vs. the artist's last tour

Matt's idea: compare each show's room with the previous tour's typical room (median capacity per show), and let the effect depend on how full that previous tour was. The curve shows how doubling the room changes the odds of a sellout. The shaded band is the range across 115 refits that each leave out one tour.

In [ ]:
coefficients = table("sellout_coefficients")
room = coefficients[(coefficients["sample"] == "history") & (coefficients["block"] == "room")]
main_coef, interaction = room.set_index("predictor")["coef"]
loo = table("room_leave_one_tour_out")
loo = loo[loo["dropped_tour"] != "none"].dropna(subset=["room_main"])

fill = np.linspace(0.75, 1.0, 101)
odds = np.exp(main_coef + interaction * (1 - fill))
refits = np.exp(loo["room_main"].to_numpy()[:, None] + loo["room_interaction"].to_numpy()[:, None] * (1 - fill))
break_even = 1 + main_coef / interaction

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.fill_between(100 * fill, refits.min(axis=0), refits.max(axis=0), color=BLUE, alpha=0.15, lw=0)
ax.plot(100 * fill, odds, color=BLUE, lw=2)
ax.axhline(1, color=MUTED, lw=1, ls="--")
ax.axvline(100 * break_even, color=MUTED, lw=1)
ax.annotate(f"break-even: {break_even:.1%} full", (100 * break_even, 0.4), xytext=(-8, 0),
            textcoords="offset points", ha="right", color=INK, fontsize=9)
ax.set_xlabel("How full the previous tour was (%)")
ax.set_ylabel("Odds multiplier per doubling of room")
ax.set_title("Upsizing only pays off after a nearly full tour", loc="left", color=INK)
plt.tight_layout()

room[["predictor", "coef", "odds_ratio", "ci_low", "ci_high", "p_value"]].round(4)

**Reading it:** after a tour that was 100% full, doubling the room multiplies the odds of selling out by about 1.4; after a 90%-full tour it lowers them to about 0.73. The break-even sits near 95% and stays between 93.5% and 98.2% whichever tour is left out. It is significant in 114 of 115 refits; the exception drops Halsey's Love and Power Tour (p = 0.063, same direction). It also holds when a sellout means 97% full.

Two limits: the room terms don't improve predictions on held-out tours, and only about 10 tours were upsized 2x or more after a tour under 95% full. It describes past bookings well; it isn't a tested forecast.

## 5. Part 2 and the combined expected fill rate

Part 2 predicts fill rate for shows that didn't sell out (fractional logit, scored by MAE). The combined model is P(sellout) x 100% + (1 - P) x Part 2, checked against a Tobit model, and scored by RMSE: MAE would reward predicting 100% for every show, since the median show sold out.

In [ ]:
fill = table("fill_rate_ladders")
fill[fill["ladder"] == "charts_first"][["test_split", "added_block", "test_tours", "mae_pts",
                                        "mae_change_pts", "change_low", "change_high"]].round(2)

In [ ]:
table("expected_fill")[["test_split", "model", "rmse_pts", "vs_baseline_low", "vs_baseline_high",
                        "mae_pts", "mean_predicted", "actual_mean"]].round(3)

**Reading it:** no model beats predicting the training average for every show, in either part. In Part 2, adding charts makes the 2019 predictions worse. The two-part model and the Tobit model agree, so this doesn't depend on how the 100% ceiling is handled.

## 6. Did COVID change what the signals mean?

One model on all pre- and post-shutdown shows, where every signal also gets a post-COVID interaction.

In [ ]:
table("covid_comparison")[["block", "signal", "odds_ratio_pre", "odds_ratio_post",
                           "interaction_p", "block_interaction_p"]].round(3)

**Reading it:** no block's effect changed (joint p-values 0.40 to 0.48).

## 7. Robustness checks

Each row reruns the controls-only and full models with one thing changed. `signal_gain` is how much the signals add (full AUC minus controls AUC).

In [ ]:
table("robustness_signal_checks").drop(columns=["shows"]).round(3)

**Reading it:**

- The picture holds under the 97% sellout definition, English-only pageviews and the worldwide sample.
- Grouped by artist, signals add nothing (+0.010). They don't carry over to artists the model hasn't seen.
- Within the training years (fit on 2017 tours, score 2018 tours), signals do help (+0.115). They work inside a period and fade on later years, which suggests the relationship drifts over time.
- Measured on the show date instead of six months before, signals help more (+0.106 on 2022-25). Buzz close to the show says more, but a buyer doesn't have it at booking.
- 26-week windows tie with 52 weeks on the inner split, so 52 weeks stays, per Matt's rule.

## 8. Right-sizing: who was upsized against the 95% rule?

Each show with a previous tour gets an odds multiplier for its room growth (downsizing isn't scored as a risk) and a flag when the room grew after a tour under the break-even.

In [ ]:
shows = table("right_sizing_shows")
print(shows.groupby("upsized_past_rule").agg(shows=("engagement_id", "size"), tours=("tour", "nunique"),
                                             sellout_rate=("sold_out", "mean")).round(3))
table("right_sizing_tours").head(10)[["artist", "tour", "shows", "median_room_ratio", "prev_tour_fill_rate",
                                      "median_odds_multiplier", "sellout_rate", "mean_fill_rate"]].round(3)

## Takeaways

1. Billboard chart and Wikipedia pageview signals, measured at booking time, barely improve on knowing the room, genre and country (test AUC 0.52 to 0.62). Neither adds anything the other doesn't.
2. Career chart history (a catalog) is the one signal that holds up, more than charting in the last year.
3. The artist's previous tour matters more than any signal, and the most useful single rule is about room size: upsizing pays off only after a tour that was about 95% full or more.
4. Same-show price and number of nights do improve predictions, but only because they're set after sales start.
5. Next: a review of which parts of the analysis add noise instead of signal (Matt's plan for after Phase 2).